# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farahhussain159-create/flyrank-ml-week1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [12]:
!git clone https://github.com/farahhussain159-create/flyrank-ml-week1.git
%cd flyrank-ml-week1

Cloning into 'flyrank-ml-week1'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 139 (delta 49), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (139/139), 1.86 MiB | 377.00 KiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/flyrank-ml-week1/flyrank-ml-week1


https://github.com/farahhussain159-create/flyrank-ml-week1.git

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [14]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(df.shape)
print(df["is_declining_label"].value_counts(normalize=True))

(30000, 45)
is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


I'm predicting is_declining_label — the same yes/no target used in my Week-4 baseline. Since this is a binary classification problem with an observed label, I start with Logistic Regression as a simple, readable baseline model, then compare it against Random Forest to see if a stronger model improves on it. This follows the method guidance in skills/training-honest-models/SKILL.md.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [15]:
feature_cols = [
    "search_volume", "competition", "competition_level", "cpc",
    "word_count", "char_count", "impressions_90d", "clicks_90d",
    "pageviews_90d", "sessions_90d", "users_90d", "engaged_sessions_90d",
    "ai_sessions_90d", "scroll_events_90d", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct"
]

X = df[feature_cols].copy()
X = pd.get_dummies(X, columns=["competition_level"], drop_first=True)
X = X.fillna(X.median(numeric_only=True))

y = df["is_declining_label"]
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train size:", X_train.shape, "Test size:", X_test.shape)
print("Unique clients in train:", df.iloc[train_idx]["client_id"].nunique())
print("Unique clients in test:", df.iloc[test_idx]["client_id"].nunique())

Train size: (23837, 22) Test size: (6163, 22)
Unique clients in train: 25
Unique clients in test: 7


I split by client_id using GroupShuffleSplit (80/20), not a random row split. This matters because the same client appears in many rows — a random split would leak that client's pattern into both train and test, making the model look better than it really is. Grouping by client_id keeps the split honest: 25 clients in train, 7 clients in test, with no overlap.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [16]:
import os
print(os.path.exists("work/outputs/baseline_action_score.csv"))
print(os.listdir("work/outputs") if os.path.exists("work/outputs") else "outputs folder missing")

False
outputs folder missing


In [17]:
# ---- Train Logistic Regression ----
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train, y_train)
logreg_proba = logreg.predict_proba(X_test)[:, 1]
logreg_pred = (logreg_proba >= 0.5).astype(int)

# ---- Train Random Forest ----
rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]
rf_pred = (rf_proba >= 0.5).astype(int)

# ---- Rebuild the Week-4 baseline rule directly (freshness_tier + search_volume, same weights) ----
tier_order = ["0-30", "31-90", "91-180", "181+"]
tier_rank = {t: i for i, t in enumerate(tier_order)}

test_baseline = df.loc[X_test.index, ["content_id", "freshness_tier", "search_volume", "is_declining_label"]].copy()

test_baseline["staleness_score"] = test_baseline["freshness_tier"].map(tier_rank)
test_baseline["staleness_score"] = 100 * test_baseline["staleness_score"] / test_baseline["staleness_score"].max()

vol = test_baseline["search_volume"]
test_baseline["volume_score"] = 100 * (vol - vol.min()) / (vol.max() - vol.min())

test_baseline["action_score"] = 0.6 * test_baseline["staleness_score"] + 0.4 * test_baseline["volume_score"]

# ---- Precision@50 helper: of the top-50 ranked items, how many are truly declining? ----
def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-np.array(scores))
    top_k_labels = np.array(y_true)[order][:k]
    return top_k_labels.mean()

baseline_p50 = precision_at_k(test_baseline["is_declining_label"], test_baseline["action_score"], k=50)
logreg_p50 = precision_at_k(y_test.values, logreg_proba, k=50)
rf_p50 = precision_at_k(y_test.values, rf_proba, k=50)

base_rate = y_test.mean()

# ---- Final comparison table ----
results = pd.DataFrame({
    "Method": ["Baseline (rule)", "Logistic Regression", "Random Forest"],
    "Precision@50": [round(baseline_p50, 3), round(logreg_p50, 3), round(rf_p50, 3)],
    "Precision (0.5 cut)": [None, round(precision_score(y_test, logreg_pred), 3), round(precision_score(y_test, rf_pred), 3)],
    "Recall (0.5 cut)": [None, round(recall_score(y_test, logreg_pred), 3), round(recall_score(y_test, rf_pred), 3)],
    "F1 (0.5 cut)": [None, round(f1_score(y_test, logreg_pred), 3), round(f1_score(y_test, rf_pred), 3)],
})

print(f"Test-set base rate (fraction actually declining): {base_rate:.3f}\n")
print(results.to_string(index=False))


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Test-set base rate (fraction actually declining): 0.511

             Method  Precision@50  Precision (0.5 cut)  Recall (0.5 cut)  F1 (0.5 cut)
    Baseline (rule)          0.38                  NaN               NaN           NaN
Logistic Regression          0.60                0.546             0.573         0.559
      Random Forest          0.52                0.565             0.742         0.642


Both models beat the baseline rule on Precision@50 (0.60 for Logistic Regression, 0.52 for
Random Forest, vs 0.38 for the baseline). This means the top-50 items each method ranks as
highest-priority are more likely to be genuinely declining content when a trained model
picks them instead of the hand-made freshness/volume rule. At the standard 0.5 cutoff,
Random Forest has the strongest F1 (0.642) and the highest recall (0.742) — it catches
more of the truly declining content, at some cost to precision (0.565). Logistic Regression
is more balanced (precision 0.546, recall 0.573). I'd recommend Random Forest if the goal is
not missing declining content, and Logistic Regression if the team wants fewer false alarms
in their action queue.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [18]:
# ---- Feature importance from Random Forest ----
importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("Top 8 features driving Random Forest predictions:\n")
print(importances.head(8))

# ---- Build an error table on the test set ----
error_df = df.loc[X_test.index, ["content_id", "client_id", "freshness_tier", "search_volume"]].copy()
error_df["actual"] = y_test.values
error_df["rf_predicted"] = rf_pred
error_df["rf_proba"] = rf_proba

# False positives: model said "declining" but it wasn't
false_positives = error_df[(error_df["rf_predicted"] == 1) & (error_df["actual"] == 0)]
# False negatives: model missed a real decline
false_negatives = error_df[(error_df["rf_predicted"] == 0) & (error_df["actual"] == 1)]

print(f"\nFalse positives (flagged as declining, but weren't): {len(false_positives)}")
print(f"False negatives (missed a real decline): {len(false_negatives)}")

print("\n3 example false positives:")
print(false_positives.sort_values("rf_proba", ascending=False).head(3).to_string(index=False))

print("\n3 example false negatives:")
print(false_negatives.sort_values("rf_proba").head(3).to_string(index=False))


Top 8 features driving Random Forest predictions:

impressions_90d           0.232686
avg_position              0.175155
content_age_days          0.154467
char_count                0.068016
word_count                0.060875
days_since_last_update    0.039335
search_volume             0.037601
ctr                       0.033847
dtype: float64

False positives (flagged as declining, but weren't): 1796
False negatives (missed a real decline): 812

3 example false positives:
          content_id         client_id freshness_tier  search_volume  actual  rf_predicted  rf_proba
content_0b47dae0c7f9 client_8527a891e2         91-180           20.0       0             1  0.819844
content_1d0963b56227 client_4e07408562         91-180           20.0       0             1  0.813477
content_41baf0722ad9 client_8527a891e2         91-180            0.0       0             1  0.809510

3 example false negatives:
          content_id         client_id freshness_tier  search_volume  actual  rf_predicted

The Random Forest leans most heavily on impressions_90d, avg_position, and content_age_days
— these three features together explain most of its decisions. This makes sense: content
that's losing visibility (impressions) and ranking poorly (avg_position) as it ages is the
clearest signature of decline.

Errors: 1796 false positives (flagged as declining when they weren't) and 812 false
negatives (missed real declines), out of 6163 test rows. Looking at examples, false
positives cluster around the 91-180 freshness tier with low search_volume — the model
seems to over-trust "getting old" as a decline signal even when volume is too small to
tell either way. Client 8527a891e2 shows up repeatedly in both false positives and false
negatives, suggesting the model doesn't generalize as well for that particular client's
content pattern — likely because client-level differences aren't captured by any feature
in this model.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.